# Infinite DOM — Training Notebook

**Runtime:** Colab T4 (smoke test) or A100 (full training)

## Pipeline
1. Install dependencies & configure remote environment
2. Load oracle training data (with CoT reasoning augmentation)
3. **SFT warmup** — Behaviour cloning with Think-then-Act format (WebAgent-R1 style)
4. **GRPO offline RL** — Per-step reward optimization with steep cliff reward
5. **Online RL (ReST)** — Agent interacts with live environment, learns from successful multi-step episodes
6. Live environment evaluation with step history (dynamic context compression)
7. Save model + plots

## 3-Phase Training Design

### Phase 1: SFT (Behaviour Cloning)
- Learn format + basic action patterns from oracle demonstrations
- Think-then-Act format: `<think>reasoning</think><answer>action</answer>`
- WebAgent-R1 showed SFT before RL is crucial — R1-Zero (no SFT) degraded

### Phase 2: GRPO (Offline Reward Optimization)
- Per-step reward refinement — sharpen action type and element selection
- Steep cliff reward: wrong action_type → 0.12, correct → 0.88+ (0.70 gap)
- Model learns precise action selection without environment interaction

### Phase 3: Online RL (Reinforced Self-Training)
- **Agent runs in the live environment for full multi-step episodes**
- Environment returns real rewards based on task graph completion
- Successful trajectories are collected and used for further fine-tuning
- THIS is the actual RL — actions affect state, rewards come from environment
- Teaches sequential strategy: "type field → click search → select result"

### Training Quality Safeguards:
- **Anti-overfitting**: Early stopping, stratified split, LoRA capacity limit
- **Anti-reward-hacking**: Steep cliff reward with hard ceilings per component
- **Anti-mode-collapse**: Diversity monitoring, temperature sampling, balanced data
- **Anti-underfitting**: 3-phase progression ensures model improves at each stage

In [5]:
# Cell 1 — Install dependencies
!pip install -q unsloth
!pip install -q "trl>=0.12.0" transformers accelerate peft
!pip install -q httpx pydantic datasets matplotlib
!pip install -q huggingface_hub
!pip install -q websockets nest_asyncio

# ── Suppress the harmless Unsloth max_new_tokens warning ──
# Unsloth internally sets max_length=32768; our code sets max_new_tokens.
# The warning "Both max_new_tokens and max_length seem to have been set"
# is noise — max_new_tokens takes precedence, which is what we want.
import warnings
warnings.filterwarnings("ignore", message="Both `max_new_tokens`.*`max_length`.*")
warnings.filterwarnings("ignore", message=".*`max_new_tokens`.*`max_length`.*")

import logging
logging.getLogger("transformers.generation.utils").setLevel(logging.ERROR)

import torch
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only'}")
if torch.cuda.is_available():
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"VRAM: {vram_gb:.1f} GB")
    if vram_gb < 14:
        print("WARNING: Less than 14GB VRAM — T4 (16GB) or better recommended")
else:
    print("WARNING: No GPU detected — training will be extremely slow")

GPU: NVIDIA A10G
VRAM: 23.9 GB


In [6]:
# Cell 2 — Configure Remote Environment & Health Check
#
# Your HF Space must be running before you start training.
# The notebook talks to the environment over HTTP for evaluation only.
# SFT and GRPO training use LOCAL oracle data — no live env needed during training.

import os
import time
import httpx

# ============================================================
# CONFIGURE YOUR HF SPACE URL HERE
# ============================================================
HF_SPACE_URL = "https://saksham1771-infinite-dom.hf.space"  # @param {type:"string"}
# ============================================================

# Strip trailing slash
HF_SPACE_URL = HF_SPACE_URL.rstrip("/")
os.environ["INFINITE_DOM_URL"] = HF_SPACE_URL

print(f"Environment URL: {HF_SPACE_URL}")
print()

# Health check with retry
def check_health(url, max_retries=3, initial_wait=5):
    """Verify the remote environment is reachable."""
    for attempt in range(max_retries):
        try:
            r = httpx.get(f"{url}/health", timeout=30)
            if r.status_code == 200:
                data = r.json()
                print(f"Connected! Server status: {data}")
                return True
            else:
                print(f"  Attempt {attempt+1}: HTTP {r.status_code}")
        except httpx.ConnectError:
            print(f"  Attempt {attempt+1}: Connection refused — is the HF Space running?")
        except httpx.TimeoutException:
            print(f"  Attempt {attempt+1}: Timeout — Space may be cold-starting (takes 1-2 min)")
        except Exception as e:
            print(f"  Attempt {attempt+1}: {type(e).__name__}: {e}")

        if attempt < max_retries - 1:
            wait = initial_wait * (2 ** attempt)
            print(f"  Retrying in {wait}s...")
            time.sleep(wait)

    print()
    print("FAILED to connect to HF Space.")
    print("Possible fixes:")
    print("  1. Check your HF_SPACE_URL is correct")
    print("  2. Make sure the Space is running (not sleeping)")
    print("  3. Visit the Space URL in your browser to wake it up")
    print()
    print("NOTE: Training (SFT + GRPO) does NOT need the live environment.")
    print("      Only Cell 10 (live evaluation) requires the connection.")
    print("      You can train now and evaluate later when the Space is ready.")
    return False

env_available = check_health(HF_SPACE_URL)
if env_available:
    # Fetch available tasks
    try:
        r = httpx.get(f"{HF_SPACE_URL}/tasks", timeout=10)
        if r.status_code == 200:
            tasks = r.json().get("tasks", [])
            print(f"\nAvailable tasks ({len(tasks)}):")
            for t in tasks:
                print(f"  Task {t['task_id']}: {t['description']}")
    except Exception:
        pass

Environment URL: https://saksham1771-infinite-dom.hf.space

Connected! Server status: {'status': 'healthy'}

Available tasks (8):
  Task 1: Booking: clean form — standard labels, no distractors
  Task 2: Booking: label drift — randomised labels + form validation
  Task 3: Booking: structural drift — shuffled fields, conditional round-trip, train selection
  Task 4: Booking: full chaos — distractors, noisy ARIA, fake buttons, 7 trains
  Task 5: E-commerce: clean store — search, filter, cart, checkout
  Task 6: E-commerce: label drift — randomised labels + checkout validation
  Task 7: E-commerce: structural drift — layout changes, more distractors, more products
  Task 8: E-commerce: full chaos — newsletter popups, fake buttons, noisy ARIA


In [7]:
# Cell 3 — Load Oracle Training Data
#
# Downloads from HuggingFace Hub (default) or uses local files.
# Downloads BOTH files:
#   - oracle_trajectories.jsonl (raw oracle data)
#   - oracle_cot.jsonl (CoT-augmented data for Think-then-Act SFT)
#
# Cell 4 will automatically use the CoT version if downloaded.

import json
from pathlib import Path
from collections import Counter

# ============================================================
# CONFIGURE DATA SOURCE
# ============================================================
DATA_SOURCE = "huggingface"  # @param ["huggingface", "local"]
HF_DATASET_REPO = "saksham1771/infinite-dom-data"  # @param {type:"string"}
LOCAL_DATA_PATH = "training/data/oracle_trajectories.jsonl"  # @param {type:"string"}
# ============================================================

COT_DATA_PATH = Path("training/data/oracle_cot.jsonl")  # Cell 4 checks this path

if DATA_SOURCE == "huggingface":
    from huggingface_hub import hf_hub_download

    # Download raw oracle data
    local_file = hf_hub_download(
        repo_id=HF_DATASET_REPO,
        filename="oracle_trajectories.jsonl",
        repo_type="dataset",
    )
    LOCAL_DATA_PATH = local_file
    print(f"Downloaded oracle_trajectories.jsonl from {HF_DATASET_REPO}")

    # Download CoT-augmented data (if it exists in the repo)
    try:
        cot_local_file = hf_hub_download(
            repo_id=HF_DATASET_REPO,
            filename="oracle_cot.jsonl",
            repo_type="dataset",
        )
        # Copy to the path Cell 4 expects
        COT_DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
        import shutil
        shutil.copy(cot_local_file, str(COT_DATA_PATH))
        print(f"Downloaded oracle_cot.jsonl from {HF_DATASET_REPO}")
        print(f"  Saved to {COT_DATA_PATH} (Cell 4 will use this for Think-then-Act SFT)")
    except Exception as e:
        print(f"No oracle_cot.jsonl in dataset repo ({e})")
        print("  Cell 4 will fall back to raw oracle data (no reasoning traces)")
        print("  To fix: run augment_cot.py locally, then upload oracle_cot.jsonl to your dataset repo")

with open(LOCAL_DATA_PATH) as f:
    records = [json.loads(line) for line in f if line.strip()]

# Validate data integrity
required_keys = {"task_id", "seed", "step", "instruction", "observation", "action"}
valid_records = []
invalid_count = 0
for r in records:
    if required_keys.issubset(r.keys()):
        action = r["action"]
        if action.get("action_type") in ("click", "type", "scroll", "wait"):
            valid_records.append(r)
        else:
            invalid_count += 1
    else:
        invalid_count += 1

records = valid_records
if invalid_count > 0:
    print(f"WARNING: Dropped {invalid_count} invalid records")

# Statistics
task_counts = Counter(r["task_id"] for r in records)
action_type_counts = Counter(r["action"]["action_type"] for r in records)
avg_obs_len = sum(len(r["observation"]) for r in records) / max(len(records), 1)

print(f"\nLoaded {len(records)} valid oracle records")
print(f"\nRecords per task:")
for tid in sorted(task_counts):
    print(f"  Task {tid}: {task_counts[tid]} records")
print(f"\nAction type distribution:")
for atype, count in action_type_counts.most_common():
    print(f"  {atype}: {count} ({100*count/len(records):.1f}%)")
print(f"\nAvg observation length: {avg_obs_len:.0f} chars")
print(f"CoT data available: {'Yes' if COT_DATA_PATH.exists() else 'No'}")

oracle_trajectories.jsonl:   0%|          | 0.00/29.7M [00:00<?, ?B/s]

Downloaded oracle_trajectories.jsonl from saksham1771/infinite-dom-data


oracle_cot.jsonl:   0%|          | 0.00/830k [00:00<?, ?B/s]

Downloaded oracle_cot.jsonl from saksham1771/infinite-dom-data
  Saved to training/data/oracle_cot.jsonl (Cell 4 will use this for Think-then-Act SFT)

Loaded 7591 valid oracle records

Records per task:
  Task 1: 1050 records
  Task 2: 1050 records
  Task 3: 1050 records
  Task 4: 720 records
  Task 5: 1050 records
  Task 6: 1050 records
  Task 7: 1050 records
  Task 8: 571 records

Action type distribution:
  click: 6373 (84.0%)
  wait: 660 (8.7%)
  type: 558 (7.4%)

Avg observation length: 3406 chars
CoT data available: Yes


In [8]:
# Cell 4 — Prepare SFT Dataset with Think-then-Act Format (WebAgent-R1)
#
# KEY CHANGE: Instead of training on raw "observation → JSON action", we train
# on "observation → <think>reasoning</think><answer>JSON action</answer>"
#
# WebAgent-R1 showed this is the single biggest performance driver for web agents.
# The model learns to REASON about what it sees before deciding what to do.
#
# If CoT-augmented data exists (from augment_cot.py), use it.
# Otherwise, fall back to raw oracle data with plain JSON responses.
#
# We pre-format texts here so Cell 6 doesn't need a formatting_func
# (avoids HuggingFace Datasets serialization issues with nested dicts).

import re
from pathlib import Path
from datasets import Dataset, DatasetDict

# --- Load model/tokenizer early (needed for apply_chat_template below) ---
from unsloth import FastLanguageModel

MODEL_ID = "unsloth/Qwen2.5-3B-Instruct"
MAX_SEQ_LEN = 4096

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_ID,
    max_seq_length=MAX_SEQ_LEN,
    dtype=None,
    load_in_4bit=True,
)

# --- Think-then-Act System Prompt (WebAgent-R1 style) ---
# IMPORTANT: This prompt must be generic — it handles BOTH booking and e-commerce tasks.
# The mode collapse fix: explicitly instruct the model to use "type" for text fields,
# not just "click" everything. The old prompt only mentioned booking.
SYSTEM_PROMPT = """You are a web agent navigating an interactive web application.
You observe an accessibility tree and must complete the given task.

First, reason about what you see and what action to take inside <think> tags.
Then, provide your action as a JSON object inside <answer> tags.

Format:
<think>
[Observe the current page state. Identify which fields need filling, what buttons are available, and decide the next step. If you see empty text fields that need values, you MUST use "type" to fill them — do NOT click on them.]
</think>
<answer>
{"action_type": "click"|"type"|"scroll"|"wait", "element_ref": "ref_id", "text_value": "text"|"", "scroll_delta": 0}
</answer>

Action rules:
- "type": Fill text fields (textbox) or select dropdown values (combobox). REQUIRES element_ref + text_value.
- "click": Press buttons or links. REQUIRES element_ref. Do NOT use click on text fields.
- "scroll": Scroll the page. Uses scroll_delta (positive=down, negative=up).
- "wait": Pause when the page is loading.

Element refs look like: inp_1, btn_2, cmb_1, lnk_3

Strategy:
1. First dismiss any cookie banners, popups, or newsletter modals (click "Accept"/"Close"/"No Thanks")
2. For booking tasks: type origin city → type destination → select class → click search → select correct train → click confirm
3. For shopping tasks: type search query → select category → click filter → click View on target product → click Add to Cart → click Checkout → type shipping name/address/city/pin/phone → click Place Order
4. Always check if fields are already filled before typing — look at value= attributes
5. If a field shows value="" it is EMPTY and needs "type" action, not "click\""""

OBS_MAX_CHARS = 3500

# --- Try loading CoT-augmented data first ---
COT_DATA_PATH = Path("training/data/oracle_cot.jsonl")
USE_COT = COT_DATA_PATH.exists()

if USE_COT:
    print("Found CoT-augmented data — using Think-then-Act format for SFT")
    with open(COT_DATA_PATH) as f:
        cot_records = [json.loads(line) for line in f if line.strip()]
    print(f"  Loaded {len(cot_records)} CoT-augmented records")
else:
    print("No CoT data found. Run: python training/augment_cot.py")
    print("Falling back to raw oracle data (no reasoning traces)")
    cot_records = None


def format_for_sft(record):
    """Convert an oracle record into a pre-formatted text string for SFT."""
    obs_text = record["observation"][:OBS_MAX_CHARS]

    step = record.get("step", 0)
    step_ctx = f"\nStep: {step}" if step > 0 else ""

    user = f"Task: {record['instruction']}\n\nAccessibility Tree:\n{obs_text}{step_ctx}"

    # Clean action dict
    action = record["action"].copy()
    for key in list(action.keys()):
        if key not in ("action_type", "element_ref", "text_value", "scroll_delta"):
            del action[key]
    action.setdefault("scroll_delta", 0)
    action.setdefault("text_value", "")
    action.setdefault("element_ref", "")

    # Use CoT response if available, otherwise plain JSON
    if USE_COT and "cot_response" in record:
        assistant_text = record["cot_response"]
    else:
        assistant_text = json.dumps(action, separators=(",", ":"))

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user},
        {"role": "assistant", "content": assistant_text},
    ]

    # Pre-format into a single text string using the tokenizer's chat template
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=False
    )

    return {
        "text": text,
        "task_id": record["task_id"],
    }


source_records = cot_records if USE_COT else records
sft_data = [format_for_sft(r) for r in source_records]

# Stratified train/eval split — ensures all task levels in eval
from collections import defaultdict
import random as stdlib_random

stdlib_random.seed(42)
by_task = defaultdict(list)
for item in sft_data:
    by_task[item["task_id"]].append(item)

train_items, eval_items = [], []
for task_id in sorted(by_task.keys()):
    items = by_task[task_id]
    stdlib_random.shuffle(items)
    split_idx = max(1, int(len(items) * 0.9))
    train_items.extend(items[:split_idx])
    eval_items.extend(items[split_idx:])

stdlib_random.shuffle(train_items)
stdlib_random.shuffle(eval_items)

# Store as plain text — no nested dicts, no serialization issues
train_ds = Dataset.from_list([{"text": item["text"]} for item in train_items])
eval_ds = Dataset.from_list([{"text": item["text"]} for item in eval_items])

print(f"\nTrain: {len(train_ds)} | Eval: {len(eval_ds)}")
print(f"Eval task distribution: {Counter(item['task_id'] for item in eval_items)}")
print(f"Format: {'Think-then-Act (CoT)' if USE_COT else 'Plain JSON'}")
print(f"Observation window: {OBS_MAX_CHARS} chars")

# Show action type balance (important for avoiding mode collapse)
atype_counts = Counter(r["action"]["action_type"] for r in source_records)
print(f"\nAction type balance in training data:")
for atype, cnt in atype_counts.most_common():
    pct = 100 * cnt / len(source_records)
    bar = "█" * int(pct / 2)
    print(f"  {atype:6s}: {cnt:4d} ({pct:4.1f}%) {bar}")
type_pct = 100 * atype_counts.get("type", 0) / len(source_records)
if type_pct < 30:
    print(f"\n⚠ WARNING: 'type' actions are only {type_pct:.0f}% of data. Risk of mode collapse to 'click'.")
    print("  Fix: regenerate oracle data with all 8 tasks (e-commerce has more type actions)")

print(f"\nSample text (first 300 chars):")
print(f"  {train_ds[0]['text'][:300]}...")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.4.8: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA A10G. Num GPUs = 1. Max memory: 22.301 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.6. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.36G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/271 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.36k [00:00<?, ?B/s]

...

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

unsloth/qwen2.5-3b-instruct-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.
Found CoT-augmented data — using Think-then-Act format for SFT
  Loaded 250 CoT-augmented records

Train: 224 | Eval: 26
Eval task distribution: Counter({2: 13, 1: 13})
Format: Think-then-Act (CoT)
Observation window: 3500 chars

Action type balance in training data:
  click :  222 (88.8%) ████████████████████████████████████████████
  type  :   28 (11.2%) █████

⚠ WARNING: 'type' actions are only 11% of data. Risk of mode collapse to 'click'.
  Fix: regenerate oracle data with all 8 tasks (e-commerce has more type actions)

Sample text (first 300 chars):
  <|im_start|>system
You are a web agent navigating an interactive web application.
You observe an accessibility tree and must complete the given task.

First, reason about what you see and what action to take inside <think> tags.
Then, provide your action as a JSON object inside <answer> tags.

Forma...


In [9]:
# Cell 5 — Load base model with Unsloth (4-bit QLoRA)
# Guide §10: TRL + Unsloth for efficiency
# Guide §16: Use QLoRA, save adapters properly

from unsloth import FastLanguageModel

MODEL_ID = "unsloth/Qwen2.5-3B-Instruct"  # Small enough for T4, capable enough for tasks
MAX_SEQ_LEN = 4096

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_ID,
    max_seq_length=MAX_SEQ_LEN,
    dtype=None,        # Auto-detect (bf16 on A100, fp16 on T4)
    load_in_4bit=True, # QLoRA — fits in T4 16GB
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,              # LoRA rank — low rank limits capacity, prevents overfitting
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                     "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,    # Unsloth recommends 0 for speed
    bias="none",
    use_gradient_checkpointing="unsloth",
)

print(f"Model loaded: {MODEL_ID}")
print(f"Trainable params: {model.print_trainable_parameters()}")

==((====))==  Unsloth 2026.4.8: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA A10G. Num GPUs = 1. Max memory: 22.301 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.6. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

unsloth/qwen2.5-3b-instruct-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


Unsloth 2026.4.8 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


Model loaded: unsloth/Qwen2.5-3B-Instruct
trainable params: 29,933,568 || all params: 3,115,872,256 || trainable%: 0.9607
Trainable params: None


In [10]:
# Cell 6 — SFT warmup training with early stopping
# Guide §3: SFT first to give the model a warm start before RL
#
# Data is already pre-formatted as plain text strings in Cell 4
# (using tokenizer.apply_chat_template), so no formatting_func needed.
#
# Anti-overfitting: EarlyStoppingCallback stops if eval loss plateaus.

from trl import SFTTrainer, SFTConfig
from transformers import EarlyStoppingCallback
import math

# Calculate steps to set good logging/eval intervals
num_samples = len(train_ds)
batch_size = 4
grad_accum = 4
effective_batch = batch_size * grad_accum
steps_per_epoch = math.ceil(num_samples / effective_batch)
total_steps = steps_per_epoch * 3  # 3 epochs

# Set eval/log intervals proportional to dataset size
# Aim for ~8-10 eval points across training
eval_interval = max(1, steps_per_epoch // 3)  # ~3 evals per epoch = ~9 total
log_interval = max(1, eval_interval // 2)     # Log twice per eval interval

print(f"Dataset: {num_samples} samples")
print(f"Steps per epoch: {steps_per_epoch} | Total steps: {total_steps}")
print(f"Eval every {eval_interval} steps | Log every {log_interval} steps")
print(f"Expected eval points: ~{total_steps // eval_interval}")
print()

sft_config = SFTConfig(
    output_dir="./sft_output",
    num_train_epochs=3,
    per_device_train_batch_size=batch_size,
    gradient_accumulation_steps=grad_accum,
    learning_rate=2e-4,
    warmup_steps=min(50, total_steps // 5),
    logging_steps=log_interval,
    save_steps=eval_interval,
    eval_strategy="steps",
    eval_steps=eval_interval,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    save_total_limit=3,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    max_seq_length=MAX_SEQ_LEN,
    dataset_text_field="text",
    seed=42,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    args=sft_config,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)

print(f"Starting SFT training...")
print(f"  Train: {len(train_ds)} | Eval: {len(eval_ds)}")
print(f"  Epochs: 3 | Early stopping patience: 3 eval checks")
print()

sft_results = trainer.train()
print(f"\nSFT complete!")
print(f"  Final training loss: {sft_results.training_loss:.4f}")
if hasattr(trainer.state, "best_metric") and trainer.state.best_metric is not None:
    print(f"  Best eval loss: {trainer.state.best_metric:.4f}")

Dataset: 224 samples
Steps per epoch: 14 | Total steps: 42
Eval every 4 steps | Log every 2 steps
Expected eval points: ~10



Unsloth: Tokenizing ["text"] (num_proc=52):   0%|          | 0/224 [00:00<?, ? examples/s]

num_proc must be <= 26. Reducing num_proc to 26 for dataset of size 26.
[datasets.arrow_dataset|WARNING]num_proc must be <= 26. Reducing num_proc to 26 for dataset of size 26.


Unsloth: Tokenizing ["text"] (num_proc=26):   0%|          | 0/26 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


...

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 224 | Num Epochs = 3 | Total steps = 42
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 4 x 1) = 16
 "-____-"     Trainable parameters = 29,933,568 of 3,115,872,256 (0.96% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,Validation Loss
4,1.706464,1.545405
8,1.267321,1.139996
12,0.979308,0.881867
16,0.765926,0.659741
20,0.520501,0.431539
24,0.314122,0.251842
28,0.181442,0.136179
32,0.094393,0.079137
36,0.066629,0.057766
40,0.054696,0.049450


Unsloth: Restored added_tokens_decoder metadata in ./sft_output/checkpoint-4/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in ./sft_output/checkpoint-8/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in ./sft_output/checkpoint-12/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in ./sft_output/checkpoint-16/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in ./sft_output/checkpoint-20/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in ./sft_output/checkpoint-24/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in ./sft_output/checkpoint-28/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in ./sft_output/checkpoint-32/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in ./sft_output/checkpoint-36/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in ./sft_output/checkpoint-40/tokenizer_confi


SFT complete!
  Final training loss: 0.6059
  Best eval loss: 0.0482


In [11]:
# Cell 7 — Save SFT checkpoint + sanity check
#
# On Colab: saves to Google Drive (survives crashes).
# On HF/other: saves locally (use HF Hub push in Cell 11 for persistence).
# Next time on Colab, skip Cells 5-7 and run Cell 7.5 (Resume) instead.

import gc

# ── Save trainer state BEFORE deleting (Cells 12/13 need it) ──
sft_log_history = list(trainer.state.log_history) if hasattr(trainer, "state") else []
sft_best_metric = getattr(trainer.state, "best_metric", None) if hasattr(trainer, "state") else None

# ── Detect platform & set checkpoint dir ──
IS_COLAB = False
try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    CHECKPOINT_DIR = "/content/drive/MyDrive/infinite-dom-checkpoints"
    IS_COLAB = True
    print("Platform: Google Colab (saving to Google Drive)")
except ImportError:
    CHECKPOINT_DIR = "./checkpoints"
    print("Platform: HuggingFace / Local (saving to ./checkpoints)")

os.makedirs(CHECKPOINT_DIR, exist_ok=True)

SFT_SAVE_PATH = f"{CHECKPOINT_DIR}/sft_lora_adapters"

model.save_pretrained(SFT_SAVE_PATH)
tokenizer.save_pretrained(SFT_SAVE_PATH)
print(f"SFT adapters saved to: {SFT_SAVE_PATH}")

# Also save to ./sft_lora_adapters for immediate use
model.save_pretrained("./sft_lora_adapters")
tokenizer.save_pretrained("./sft_lora_adapters")
print("SFT adapters also saved locally: ./sft_lora_adapters")

# ── Free SFT trainer memory ──
del trainer
gc.collect()
torch.cuda.empty_cache()
print(f"Freed SFT trainer memory. GPU free: {torch.cuda.mem_get_info()[0]/1e9:.1f} GB")

# ── Sanity check ──
def parse_model_output(text):
    """Extract JSON action from <answer> tags or raw JSON."""
    text = text.strip()
    if "<answer>" in text:
        start = text.index("<answer>") + len("<answer>")
        end = text.index("</answer>") if "</answer>" in text else len(text)
        text = text[start:end].strip()
    if text.startswith("```"):
        text = text.split("```")[1]
        if text.startswith("json"):
            text = text[4:]
    return json.loads(text)

FastLanguageModel.for_inference(model)

test_observations = [
    """Task: Book a Sleeper ticket from Delhi to Mumbai

Accessibility Tree:
[ref=mn_1 role=main]
  [ref=frm_1 role=form name="Search Trains"]
    [ref=inp_1 role=textbox name="From" value=""]
    [ref=inp_2 role=textbox name="To" value=""]
    [ref=cmb_1 role=combobox name="Class" value="-- Select --"]
    [ref=btn_1 role=button name="Search"]""",

    """Task: Book a Chair Car ticket from Pune to Jaipur

Accessibility Tree:
[ref=mn_1 role=main]
  [ref=btn_2 role=button name="Accept" description="cookie banner"]
  [ref=frm_1 role=form name="Booking"]
    [ref=cmb_1 role=combobox name="Seat Type" value="-- Choose --"]
    [ref=inp_1 role=textbox name="Origin" value=""]
    [ref=inp_2 role=textbox name="Destination" value=""]
    [ref=btn_1 role=button name="Go"]""",
]

print("\nSFT Sanity Check:")
print("=" * 60)

for i, test_input in enumerate(test_observations):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": test_input},
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    outputs = model.generate(**inputs, max_new_tokens=200, temperature=0.1)
    response = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)

    print(f"\nTest {i+1} ({len(response.split())} tokens):")
    print(f"  {response.strip()[:250]}")

    try:
        parsed = parse_model_output(response)
        print(f"  Parsed: {parsed}")
        print(f"  Status: PASS")
    except Exception:
        print(f"  Status: FAIL (could not parse JSON)")

print(f"\n{'=' * 60}")
print("If PASS: proceed to Cell 8 (GRPO)")
print("If FAIL: check Cell 4 data format or increase SFT epochs")

Platform: HuggingFace / Local (saving to ./checkpoints)


Unsloth: Restored added_tokens_decoder metadata in ./checkpoints/sft_lora_adapters/tokenizer_config.json.


SFT adapters saved to: ./checkpoints/sft_lora_adapters


Unsloth: Restored added_tokens_decoder metadata in ./sft_lora_adapters/tokenizer_config.json.


SFT adapters also saved locally: ./sft_lora_adapters
Freed SFT trainer memory. GPU free: 18.7 GB

SFT Sanity Check:


/home/user/miniconda/lib/python3.10/site-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/home/user/miniconda/lib/python3.10/site-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)



Test 1 (36 tokens):
  <think>
I see empty textboxes for From (Delhi) and To (Mumbai). I need to type "Delhi" into inp_1 and "Mumbai" into inp_2. btn_1 is disabled so I can't click it yet.
</think>
<answer>
{"action_type":"type","element_ref":"inp_1","text_value":"Delhi","
  Parsed: {'action_type': 'type', 'element_ref': 'inp_1', 'text_value': 'Delhi', 'scroll_delta': 0}
  Status: PASS

Test 2 (35 tokens):
  <think>
I've seen btn_2 (accept), frm_1 (booking form), cmb_1 (seat type), inp_1 (origin), inp_2 (destination), btn_1 (search). Origin and destination fields are empty; I need to type values here.
</think>
<answer>
{"action_type":"type","element_ref"
  Parsed: {'action_type': 'type', 'element_ref': 'inp_1', 'text_value': 'Pune', 'scroll_delta': 0}
  Status: PASS

If PASS: proceed to Cell 8 (GRPO)
If FAIL: check Cell 4 data format or increase SFT epochs


In [ ]:
# Cell 7.5 — RESUME POINT: Load saved model from checkpoint
#
# ╔══════════════════════════════════════════════════════════════╗
# ║  USE THIS CELL ONLY WHEN RESUMING AFTER A SESSION CRASH.   ║
# ║  Skip Cells 5, 6, 7 — this loads the saved SFT/GRPO model.║
# ║                                                              ║
# ║  Resume flow: Cell 1 → 2 → 3 → 4 → THIS → 8              ║
# ╚══════════════════════════════════════════════════════════════╝

import gc
import os

# ── Detect platform & set checkpoint dir ──
IS_COLAB = False
try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    CHECKPOINT_DIR = "/content/drive/MyDrive/infinite-dom-checkpoints"
    IS_COLAB = True
    print("Platform: Google Colab (loading from Google Drive)")
except ImportError:
    CHECKPOINT_DIR = "./checkpoints"
    print("Platform: HuggingFace / Local (loading from ./checkpoints)")

# Check what's available (tasks 1-8)
available = []
if os.path.exists(f"{CHECKPOINT_DIR}/sft_lora_adapters"):
    available.append("SFT")
for tid in range(1, 9):
    if os.path.exists(f"{CHECKPOINT_DIR}/grpo_after_task_{tid}"):
        available.append(f"GRPO-Task{tid}")

print(f"Available checkpoints: {available if available else 'NONE'}")

if not available:
    print("\nNo checkpoints found. You need to run Cells 5-7 first (full SFT training).")
else:
    from unsloth import FastLanguageModel

    MAX_SEQ_LEN = 4096

    # Load the LATEST checkpoint (highest completed GRPO task)
    load_path = None
    for tid in range(8, 0, -1):
        if os.path.exists(f"{CHECKPOINT_DIR}/grpo_after_task_{tid}"):
            load_path = f"{CHECKPOINT_DIR}/grpo_after_task_{tid}"
            break
    if load_path is None:
        load_path = f"{CHECKPOINT_DIR}/sft_lora_adapters"

    print(f"\nLoading from: {load_path}")

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=load_path,
        max_seq_length=MAX_SEQ_LEN,
        dtype=None,
        load_in_4bit=True,
    )

    # Track which GRPO tasks are done
    COMPLETED_TASKS = set()
    for tid in range(1, 9):
        if os.path.exists(f"{CHECKPOINT_DIR}/grpo_after_task_{tid}"):
            COMPLETED_TASKS.add(tid)

    # Initialize variables that Cells 11-13 expect (skipped SFT in resume flow)
    sft_log_history = []
    sft_best_metric = None

    print(f"Model loaded!")
    print(f"GRPO tasks done: {sorted(COMPLETED_TASKS) if COMPLETED_TASKS else 'None'}")
    remaining = [t for t in range(1, 9) if t not in COMPLETED_TASKS]
    print(f"Cell 8 will train: {remaining if remaining else 'All done!'}")
    print(f"NOTE: sft_log_history/sft_best_metric unavailable (resumed from checkpoint)")

    gc.collect()
    torch.cuda.empty_cache()
    print(f"GPU free: {torch.cuda.mem_get_info()[0]/1e9:.1f} GB")

In [12]:
# Cell 8 — GRPO RL Training (8-Task Curriculum with Steep Cliff Reward)
#
# REWARD V2 — Steep cliffs to fix mode collapse:
#   Wrong action_type → max 0.12 (was ~0.40)
#   Right type, wrong element → max 0.25 (was ~0.45)
#   Right type + element → 0.58-0.92
#   Gap between bad/good: 0.70 (was 0.20) — GRPO can now learn
#
# Speed settings:
#   400 records/task + batch=2 + grad_accum=8 + num_gen=2 → ~50 steps/task
#   8 tasks × 50 steps ≈ 45-60 min on A100

from trl import GRPOTrainer, GRPOConfig
import re
import gc

FastLanguageModel.for_training(model)

SCORE_MIN, SCORE_MAX = 0.01, 0.99
VALID_ACTION_TYPES = {"click", "type", "scroll", "wait"}
VALID_REF_PATTERN = re.compile(r"^(inp|btn|cmb|lnk|frm|hdg|mn|nav|chk|rad|opt|txt|lst|el)_\d+$")

# Use CHECKPOINT_DIR from Cell 7 / 7.5 (platform-agnostic)
if "CHECKPOINT_DIR" not in dir():
    CHECKPOINT_DIR = "./checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

if "COMPLETED_TASKS" not in dir():
    COMPLETED_TASKS = set()

MAX_RECORDS_PER_TASK = 400  # 400 / (batch=2 * grad_accum=8) = 50 steps per task

TASK_LABELS = {
    1: "Booking: Clean Form",
    2: "Booking: Label Drift + Validation",
    3: "Booking: Structural Drift + Train Selection",
    4: "Booking: Full Chaos",
    5: "E-commerce: Clean Store",
    6: "E-commerce: Label Drift + Validation",
    7: "E-commerce: Structural Drift + Distractors",
    8: "E-commerce: Full Chaos",
}


def _parse_action_from_text(text):
    """Extract JSON action dict from model completion text."""
    text = text.strip()
    action_text = text
    if "<answer>" in text:
        try:
            a_s = text.index("<answer>") + 8
            a_e = text.index("</answer>") if "</answer>" in text else len(text)
            action_text = text[a_s:a_e].strip()
        except ValueError:
            pass
    if action_text.startswith("```"):
        parts = action_text.split("```")
        if len(parts) > 1:
            action_text = parts[1].lstrip("json").strip()
    try:
        parsed = json.loads(action_text)
        return parsed if isinstance(parsed, dict) else {}
    except (json.JSONDecodeError, TypeError):
        return {}


def oracle_comparison_reward(completions, oracle_action_type=None,
                              oracle_element_ref=None, oracle_text_value=None,
                              **kwargs):
    """
    Steep cliff reward — forces GRPO to learn correct actions.

    Scoring (early-exit cliffs):
      No parseable action         → 0.01 (floor)
      Wrong action_type           → 0.03-0.12 (HARD ceiling)
      Right type, wrong element   → 0.18-0.25
      Right type + right element  → 0.58-0.92
      Perfect match               → 0.88-0.92

    The 0.70 gap between wrong-type (0.12) and right-type+element (0.82+)
    gives GRPO a massive gradient signal to learn correct action selection.
    """
    rewards = []

    for i, completion in enumerate(completions):
        if isinstance(completion, list):
            text = completion[-1]["content"] if completion else ""
        else:
            text = str(completion)
        text = text.strip()

        o_atype = oracle_action_type[i] if oracle_action_type else ""
        o_ref = oracle_element_ref[i] if oracle_element_ref else ""
        o_tval = oracle_text_value[i] if oracle_text_value else ""

        predicted = _parse_action_from_text(text)
        p_atype = predicted.get("action_type", "")
        p_ref = predicted.get("element_ref", "")
        p_tval = predicted.get("text_value", "")

        has_format = "<think>" in text and "<answer>" in text and predicted

        # ── CLIFF 0: No parseable action → floor ──
        if not predicted:
            rewards.append(SCORE_MIN)
            continue

        # ── CLIFF 1: Wrong action_type → max 0.12 ──
        if p_atype != o_atype:
            score = 0.08 if has_format else 0.03
            if p_atype in VALID_ACTION_TYPES:
                score += 0.04
            rewards.append(max(SCORE_MIN, min(SCORE_MAX, score)))
            continue

        # ── CLIFF 2: Right type, wrong element → max 0.25 ──
        if o_ref and p_ref != o_ref:
            score = 0.18
            if p_ref and p_ref.split("_")[0] == o_ref.split("_")[0]:
                score = 0.23  # Same element category (inp vs inp)
            if has_format:
                score += 0.02
            rewards.append(max(SCORE_MIN, min(SCORE_MAX, score)))
            continue

        # ── RIGHT TYPE + RIGHT ELEMENT: 0.58 base + up to 0.34 ──
        score = 0.58
        if has_format:
            score += 0.04

        # Text value match (for "type" actions)
        if o_atype == "type" and o_tval:
            if p_tval.strip().lower() == o_tval.strip().lower():
                score += 0.30  # Perfect text → 0.92
            elif p_tval and o_tval.lower() in p_tval.lower():
                score += 0.15  # Partial text → 0.77
            elif p_tval:
                score += 0.02  # Some text → 0.64
            # No text_value for type action → stays at 0.58-0.62
        else:
            score += 0.30  # Click/scroll don't need text → full credit

        rewards.append(max(SCORE_MIN, min(SCORE_MAX, score)))

    return rewards


# ─── Config: 50 steps per task ───
# batch=2 * grad_accum=8 = effective batch 16
# 400 records / 16 = 50 steps per task
# 8 tasks × 50 steps ≈ 45-60 min on A100
grpo_config = GRPOConfig(
    output_dir="./grpo_output",
    num_train_epochs=1,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,   # Was 4 → doubled for fewer steps
    learning_rate=5e-6,
    logging_steps=5,
    save_steps=500,
    max_completion_length=150,
    num_generations=2,
    temperature=0.7,
    report_to="none",
    seed=42,
)

# ─── 8-Task Curriculum training with steep cliff reward ───
reward_history = []

for task_id in range(1, 9):
    if task_id in COMPLETED_TASKS:
        print(f"\n>>> Task {task_id}: ALREADY DONE (from checkpoint) — skipping")
        continue

    label = TASK_LABELS.get(task_id, f"Task {task_id}")
    print(f"\n{'='*60}")
    print(f"GRPO Training — Task {task_id}: {label}")
    print(f"{'='*60}")

    task_records = [r for r in records if r["task_id"] == task_id]
    if not task_records:
        print(f"  No oracle data for task {task_id} — skipping")
        print(f"  (Regenerate data: python training/generate_oracle_data.py 30 {task_id})")
        continue

    # Cap records per task to keep training fast
    if len(task_records) > MAX_RECORDS_PER_TASK:
        import random as _rng
        _rng.seed(42 + task_id)
        task_records = _rng.sample(task_records, MAX_RECORDS_PER_TASK)
        print(f"  Subsampled to {MAX_RECORDS_PER_TASK} records (from {len([r for r in records if r['task_id'] == task_id])})")

    # Build prompts AND oracle actions for the dataset
    prompts = []
    oracle_atypes = []
    oracle_refs = []
    oracle_tvals = []

    for r in task_records:
        obs_text = r["observation"][:OBS_MAX_CHARS]
        step = r.get("step", 0)
        step_ctx = f"\nStep: {step}" if step > 0 else ""
        user_msg = f"Task: {r['instruction']}\n\nAccessibility Tree:\n{obs_text}{step_ctx}"
        msgs = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_msg},
        ]
        prompts.append(msgs)

        action = r["action"]
        oracle_atypes.append(action.get("action_type", ""))
        oracle_refs.append(action.get("element_ref", ""))
        oracle_tvals.append(action.get("text_value", ""))

    prompt_dataset = Dataset.from_dict({
        "prompt": prompts,
        "oracle_action_type": oracle_atypes,
        "oracle_element_ref": oracle_refs,
        "oracle_text_value": oracle_tvals,
    })

    eff_batch = grpo_config.per_device_train_batch_size * grpo_config.gradient_accumulation_steps
    total_steps = (len(prompts) * grpo_config.num_train_epochs) // eff_batch
    print(f"  Records: {len(prompts)} | Epochs: {grpo_config.num_train_epochs}")
    print(f"  Estimated steps: ~{total_steps} | num_generations: {grpo_config.num_generations}")
    print(f"  LR: {grpo_config.learning_rate:.2e} | Temp: {grpo_config.temperature}")
    print(f"  GPU free before: {torch.cuda.mem_get_info()[0]/1e9:.1f} GB")

    atype_dist = Counter(oracle_atypes)
    print(f"  Oracle action dist: {dict(atype_dist)}")

    grpo_trainer = GRPOTrainer(
        model=model,
        tokenizer=tokenizer,
        reward_funcs=[oracle_comparison_reward],
        train_dataset=prompt_dataset,
        args=grpo_config,
    )

    result = grpo_trainer.train()

    # ── Extract metrics ──
    avg_reward = None
    final_reward_std = None
    reward_trend = []
    if hasattr(grpo_trainer, "state") and grpo_trainer.state.log_history:
        reward_logs = [h["reward"] for h in grpo_trainer.state.log_history if "reward" in h]
        std_logs = [h["reward_std"] for h in grpo_trainer.state.log_history if "reward_std" in h]
        if reward_logs:
            avg_reward = sum(reward_logs) / len(reward_logs)
            reward_trend = reward_logs
        if std_logs:
            final_reward_std = std_logs[-1]

    loss_val = result.training_loss
    print(f"\n  Results:")
    print(f"    Training loss: {loss_val:.4f}")
    if avg_reward is not None:
        print(f"    Avg reward: {avg_reward:.4f}")
    if final_reward_std is not None:
        health = "HEALTHY" if final_reward_std > 0.05 else "OK" if final_reward_std > 0.02 else "LOW"
        print(f"    Final reward_std: {final_reward_std:.4f} ({health})")
    if len(reward_trend) >= 4:
        first_q = sum(reward_trend[:len(reward_trend)//4]) / (len(reward_trend)//4)
        last_q = sum(reward_trend[-len(reward_trend)//4:]) / (len(reward_trend)//4)
        direction = "IMPROVING" if last_q > first_q + 0.02 else "FLAT" if abs(last_q - first_q) < 0.02 else "DECLINING"
        print(f"    Reward trend: {first_q:.3f} → {last_q:.3f} ({direction})")

    reward_history.append({
        "task_id": task_id,
        "label": label,
        "loss": loss_val,
        "avg_reward": avg_reward,
        "final_std": final_reward_std,
    })

    # ── Save checkpoint (crash-safe) ──
    save_path = f"{CHECKPOINT_DIR}/grpo_after_task_{task_id}"
    model.save_pretrained(save_path)
    tokenizer.save_pretrained(save_path)
    COMPLETED_TASKS.add(task_id)
    print(f"    Saved checkpoint: {save_path}")

    # ── Free memory before next task ──
    del grpo_trainer
    gc.collect()
    torch.cuda.empty_cache()
    print(f"    GPU free after cleanup: {torch.cuda.mem_get_info()[0]/1e9:.1f} GB")

    # Softer LR decay for harder tasks
    grpo_config.learning_rate *= 0.9

print(f"\n{'='*60}")
print("GRPO 8-task curriculum training complete!")
print(f"{'='*60}")
print(f"\nReward cliff design:")
print(f"  Wrong action_type  → 0.03-0.12 (HARD ceiling)")
print(f"  Right type, wrong  → 0.18-0.25")
print(f"  Right type+element → 0.58-0.92")
print(f"  Gap: 0.70 — GRPO has strong learning signal")
print()
print(f"Diagnostics:")
print(f"  reward INCREASING + std > 0.05 → model is learning")
print(f"  reward_std → 0 → mode collapse, raise temperature")
print()
for rh in reward_history:
    r_str = f"  reward={rh['avg_reward']:.4f}" if rh.get("avg_reward") else ""
    s_str = f"  std={rh['final_std']:.4f}" if rh.get("final_std") else ""
    print(f"  Task {rh['task_id']} ({rh['label']}): loss={rh['loss']:.4f}{r_str}{s_str}")


GRPO Training — Task 1: Booking: Clean Form
  Subsampled to 400 records (from 1050)
  Records: 400 | Epochs: 1
  Estimated steps: ~25 | num_generations: 2
  LR: 5.00e-06 | Temp: 0.7
  GPU free before: 18.7 GB
  Oracle action dist: {'click': 366, 'type': 34}


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 400 | Num Epochs = 1 | Total steps = 50
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 8 x 1) = 16
 "-____-"     Trainable parameters = 29,933,568 of 3,115,872,256 (0.96% trained)
/home/user/miniconda/lib/python3.10/site-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/home/user/miniconda/lib/python3.10/site-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10

Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,reward,reward_std,completions / mean_length,completions / min_length,completions / max_length,completions / clipped_ratio,completions / mean_terminated_length,completions / min_terminated_length,completions / max_terminated_length,kl,rewards / oracle_comparison_reward / mean,rewards / oracle_comparison_reward / std
5,0.021727,0.489250,0.097934,77.375000,56.200000,123.200000,0.037500,74.680597,56.200000,111.600000,1.915217,0.489250,0.357421
10,0.010063,0.507125,0.025279,72.000000,55.800000,126.200000,0.037500,68.952501,55.800000,91.800000,1.650563,0.507125,0.354011
15,0.001464,0.563000,0.041012,69.062500,57.000000,94.400000,0.000000,69.062500,57.000000,94.400000,1.716535,0.563000,0.339377
20,0.105310,0.643375,0.031289,69.675000,55.000000,103.000000,0.000000,69.675000,55.000000,103.000000,82.862769,0.643375,0.341967
25,0.004893,0.540500,0.032880,68.525000,56.400000,97.000000,0.012500,67.508334,56.400000,83.200000,1.701322,0.540500,0.353874
30,0.014234,0.511625,0.012551,71.100000,57.800000,108.400000,0.012500,70.105833,57.800000,101.800000,1.714634,0.511625,0.302580
35,0.004103,0.550750,0.009192,69.400000,56.000000,108.600000,0.000000,69.400000,56.000000,108.600000,1.839403,0.550750,0.361565
40,0.012787,0.595750,0.027577,71.087500,58.400000,107.800000,0.012500,70.122501,58.400000,98.800000,2.238990,0.595750,0.319909
45,0.003576,0.581375,0.013258,67.700000,58.000000,81.800000,0.000000,67.700000,58.000000,81.800000,1.757816,0.581375,0.338092


/home/user/miniconda/lib/python3.10/site-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/home/user/miniconda/lib/python3.10/site-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/home/user/miniconda/lib/python3.10/site-packages/transformers/modeling_attn_mask_utils.py:172: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers 

KeyboardInterrupt: 

In [13]:
# Cell 8.5 — Online RL: Reinforced Self-Training (ReST) with Live Environment
#
# THIS IS THE ACTUAL RL PART — the model interacts with the live environment,
# takes multi-step actions, observes consequences, and learns from success.
#
# Method: ReST (Reinforced Self-Training) / Expert Iteration
#   1. Run model in live environment for full multi-step episodes
#   2. Environment returns rewards based on task graph completion
#   3. Keep trajectories where agent completed ≥1 task node (success signal)
#   4. Fine-tune on successful (observation → action) pairs
#   5. This teaches sequential strategy — not just single-action selection
#
# Why this matters vs Cell 8 (GRPO):
#   Cell 8 = offline, single-step, oracle comparison (reward-weighted SFT)
#   Cell 8.5 = ONLINE, multi-step, environment interaction (actual RL)
#   The model learns "type in field → page changes → click search → results appear"
#
# Requires: HF Space running with fixed Dockerfile (playwright==1.48.0)
# Time: ~15-20 min on A100 (5 episodes × 8 tasks × ~25 steps each)

import asyncio
import nest_asyncio
import websockets
import httpx

nest_asyncio.apply()

INFINITE_DOM_URL = os.environ.get("INFINITE_DOM_URL", HF_SPACE_URL)
WS_URL = INFINITE_DOM_URL.replace("https://", "wss://").replace("http://", "ws://") + "/ws"

# ── Check environment connectivity ──
print("Online RL: Checking live environment...")
env_ok = False
try:
    r = httpx.get(f"{INFINITE_DOM_URL}/health", timeout=30)
    if r.status_code == 200:
        print(f"  Connected: {INFINITE_DOM_URL}")
        env_ok = True
except Exception as e:
    print(f"  Connection failed: {e}")

if not env_ok:
    print("\nHF Space not available — skipping online RL.")
    print("This cell requires the live environment. Deploy first, then rerun.")
    print("Continuing to Cell 9 (offline quality check)...")
    online_rl_stats = {"skipped": True, "reason": "env_unavailable"}
else:
    EPISODES_PER_TASK = 5
    MAX_EPISODE_STEPS = 25
    MIN_NODES_FOR_SUCCESS = 1  # Keep episodes that complete at least 1 node

    FastLanguageModel.for_inference(model)

    def model_policy_explore(obs, step_history=None):
        """Model policy with higher temperature for exploration during RL."""
        tree = obs.get("a11y_tree", "")[:OBS_MAX_CHARS]
        instruction = obs.get("task_instruction", "")
        progress = obs.get("task_progress", [])

        history_text = ""
        if step_history:
            recent = step_history[-5:]
            history_text = "Previous actions:\n" + "\n".join(f"  {s}" for s in recent) + "\n\n"

        user_msg = (
            f"Task: {instruction}\n\n"
            f"{history_text}"
            f"Accessibility Tree:\n{tree}\n\n"
            f"Completed: {progress}"
        )

        msgs = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_msg},
        ]
        prompt = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
        with torch.no_grad():
            outputs = model.generate(
                **inputs, max_new_tokens=150, temperature=0.5, do_sample=True
            )
        response = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)

        raw_text = response.strip()
        try:
            parsed = parse_model_output(response)
            return parsed, raw_text
        except (json.JSONDecodeError, ValueError):
            return {"action_type": "wait", "element_ref": "", "text_value": "", "scroll_delta": 0}, raw_text


    async def collect_episode(task_id, seed):
        """Run one full episode, return trajectory + completion info."""
        trajectory = []
        try:
            async with websockets.connect(WS_URL, open_timeout=120, close_timeout=10) as ws:
                # Reset
                await ws.send(json.dumps({
                    "type": "reset",
                    "data": {"task_id": task_id, "seed": seed}
                }))
                raw = await asyncio.wait_for(ws.recv(), timeout=120)
                resp = json.loads(raw)

                if resp.get("type") == "error":
                    return [], 0, 0

                obs = resp["data"]["observation"]
                done = False
                total_nodes = obs.get("metadata", {}).get("total_nodes", 5)
                step_history = []

                while not done and len(trajectory) < MAX_EPISODE_STEPS:
                    # Model chooses action
                    action, raw_text = model_policy_explore(obs, step_history)

                    # Record (observation, model_output) BEFORE stepping
                    trajectory.append({
                        "obs": obs,
                        "action": action,
                        "raw_text": raw_text,
                        "step_history": list(step_history),
                    })

                    # Step environment — THIS IS THE RL INTERACTION
                    await ws.send(json.dumps({"type": "step", "data": action}))
                    raw = await asyncio.wait_for(ws.recv(), timeout=30)
                    resp = json.loads(raw)

                    if resp.get("type") == "error":
                        break

                    obs = resp["data"]["observation"]
                    done = resp["data"].get("done", False)

                    # Track history
                    atype = action.get("action_type", "?")
                    ref = action.get("element_ref", "")
                    tval = action.get("text_value", "")
                    desc = f"{atype}"
                    if ref:
                        desc += f" {ref}"
                    if tval:
                        desc += f" '{tval[:20]}'"
                    step_history.append(desc)

                completed = len(obs.get("task_progress", []))
                return trajectory, completed, total_nodes

        except Exception as e:
            return [], 0, 0


    # ── Collect trajectories from all 8 tasks ──
    print(f"\nCollecting online trajectories: {EPISODES_PER_TASK} episodes x 8 tasks")
    print(f"Max steps per episode: {MAX_EPISODE_STEPS}")
    print(f"Success threshold: {MIN_NODES_FOR_SUCCESS}+ nodes completed")
    print("=" * 60)

    all_success_pairs = []
    online_rl_stats = {"episodes": 0, "successes": 0, "total_pairs": 0, "by_task": {}}

    for task_id in range(1, 9):
        label = TASK_LABELS.get(task_id, f"Task {task_id}")
        task_pairs = []
        task_successes = 0

        for seed in range(EPISODES_PER_TASK):
            trajectory, completed, total = asyncio.get_event_loop().run_until_complete(
                collect_episode(task_id, seed=seed + 200)
            )
            online_rl_stats["episodes"] += 1

            is_success = completed >= MIN_NODES_FOR_SUCCESS
            if is_success:
                task_successes += 1
                online_rl_stats["successes"] += 1

                # Convert successful trajectory to SFT training pairs
                for step_data in trajectory:
                    obs = step_data["obs"]
                    action = step_data["action"]
                    raw_text = step_data["raw_text"]
                    hist = step_data["step_history"]

                    tree = obs.get("a11y_tree", "")[:OBS_MAX_CHARS]
                    instruction = obs.get("task_instruction", "")
                    progress = obs.get("task_progress", [])

                    history_text = ""
                    if hist:
                        recent = hist[-5:]
                        history_text = "Previous actions:\n" + "\n".join(f"  {s}" for s in recent) + "\n\n"

                    user_msg = (
                        f"Task: {instruction}\n\n"
                        f"{history_text}"
                        f"Accessibility Tree:\n{tree}\n\n"
                        f"Completed: {progress}"
                    )

                    # Use model's own output as target (self-training)
                    if "<answer>" in raw_text:
                        assistant_text = raw_text
                    else:
                        action_json = json.dumps({
                            "action_type": action.get("action_type", "wait"),
                            "element_ref": action.get("element_ref", ""),
                            "text_value": action.get("text_value", ""),
                            "scroll_delta": action.get("scroll_delta", 0),
                        })
                        assistant_text = f"<think>Executing next step.</think>\n<answer>\n{action_json}\n</answer>"

                    messages = [
                        {"role": "system", "content": SYSTEM_PROMPT},
                        {"role": "user", "content": user_msg},
                        {"role": "assistant", "content": assistant_text},
                    ]
                    text = tokenizer.apply_chat_template(
                        messages, tokenize=False, add_generation_prompt=False
                    )
                    task_pairs.append({"text": text})

            status = f"{completed}/{total} nodes" + (" *" if is_success else "")
            print(f"  Task {task_id} seed {seed}: {status}")

        all_success_pairs.extend(task_pairs)
        online_rl_stats["by_task"][task_id] = {
            "successes": task_successes,
            "pairs": len(task_pairs),
        }
        print(f"  => {task_successes}/{EPISODES_PER_TASK} successful, {len(task_pairs)} training pairs")

    online_rl_stats["total_pairs"] = len(all_success_pairs)
    print(f"\n{'='*60}")
    print(f"Online RL collection complete:")
    print(f"  Episodes: {online_rl_stats['episodes']}")
    print(f"  Successful: {online_rl_stats['successes']}")
    print(f"  Training pairs: {online_rl_stats['total_pairs']}")

    # ── Fine-tune on successful trajectories ──
    if len(all_success_pairs) >= 20:
        from trl import SFTTrainer, SFTConfig

        FastLanguageModel.for_training(model)

        online_ds = Dataset.from_list(all_success_pairs)

        online_config = SFTConfig(
            output_dir="./online_rl_output",
            num_train_epochs=2,
            per_device_train_batch_size=2,
            gradient_accumulation_steps=4,
            learning_rate=1e-5,  # Low LR — refinement, not retraining
            logging_steps=5,
            save_steps=500,
            max_seq_length=MAX_SEQ_LEN,
            dataset_text_field="text",
            fp16=not torch.cuda.is_bf16_supported(),
            bf16=torch.cuda.is_bf16_supported(),
            seed=42,
            report_to="none",
        )

        online_trainer = SFTTrainer(
            model=model,
            tokenizer=tokenizer,
            train_dataset=online_ds,
            args=online_config,
        )

        print(f"\nOnline RL fine-tuning on {len(online_ds)} successful trajectory pairs...")
        online_result = online_trainer.train()
        online_rl_stats["sft_loss"] = online_result.training_loss
        print(f"  Loss: {online_result.training_loss:.4f}")

        # Save checkpoint
        save_path = f"{CHECKPOINT_DIR}/online_rl_final"
        model.save_pretrained(save_path)
        tokenizer.save_pretrained(save_path)
        print(f"  Checkpoint: {save_path}")

        del online_trainer
        gc.collect()
        torch.cuda.empty_cache()

        print(f"\nOnline RL complete! Model has now learned from:")
        print(f"  1. Oracle demonstrations (SFT)")
        print(f"  2. Per-step reward optimization (GRPO)")
        print(f"  3. Its own successful multi-step episodes (Online RL) <-- NEW")
    else:
        print(f"\nOnly {len(all_success_pairs)} pairs — not enough for fine-tuning.")
        print("The model needs more offline training before online RL can help.")
        print("This is normal for early iterations — the SFT+GRPO model may not")
        print("complete enough nodes yet. Try again after improving offline training.")

Online RL: Checking live environment...
  Connected: https://saksham1771-infinite-dom.hf.space

Max steps per episode: 25
Success threshold: 1+ nodes completed
  Task 1 seed 0: 0/5 nodes
  Task 1 seed 1: 0/5 nodes
  Task 1 seed 2: 0/5 nodes


KeyboardInterrupt: 

In [14]:
# Cell 9 — Post-RL Quality Check: reasoning + diversity + correctness
#
# Tests BOTH booking and e-commerce tasks to verify:
# 1. Does the model produce <think>...</think><answer>...</answer> format?
# 2. Is the reasoning relevant (mentions elements, task terms)?
# 3. Are actions still valid JSON with correct action_type?
# 4. Is there diversity (no mode collapse)?
# 5. Does the model use "type" where appropriate (not just "click" everything)?

FastLanguageModel.for_inference(model)

quality_tests = [
    # --- Booking tasks ---
    ("Booking Easy", """Task: Book a Sleeper ticket from Delhi to Mumbai

Accessibility Tree:
[ref=mn_1 role=main]
  [ref=frm_1 role=form name="Search Trains"]
    [ref=inp_1 role=textbox name="From" value=""]
    [ref=inp_2 role=textbox name="To" value=""]
    [ref=cmb_1 role=combobox name="Class" value="-- Select --"]
    [ref=btn_1 role=button name="Search"]"""),

    ("Booking Medium", """Task: Book an AC 2 Tier ticket from Hyderabad to Pune

Accessibility Tree:
[ref=mn_1 role=main]
  [ref=frm_1 role=form name="Journey Planner"]
    [ref=inp_1 role=textbox name="Starting Point" value=""]
    [ref=inp_2 role=textbox name="Going To" value=""]
    [ref=cmb_1 role=combobox name="Coach Class" value="-- Pick --"]
    [ref=btn_1 role=button name="Check Availability"]"""),

    ("Booking Hard", """Task: Book a Chair Car ticket from Kolkata to Lucknow

Accessibility Tree:
[ref=mn_1 role=main]
  [ref=btn_99 role=button name="Accept" description="cookie banner"]
  [ref=sec_1 role=region name="Special Offer"]
    [ref=btn_98 role=button name="Claim Now"]
  [ref=frm_1 role=form name="Travel Booking"]
    [ref=cmb_1 role=combobox name="Seat Type" value="-- Choose --"]
    [ref=inp_1 role=textbox name="Origin" value=""]
    [ref=inp_2 role=textbox name="Destination" value=""]
    [ref=btn_1 role=button name="Look Up Trains"]"""),

    # --- E-commerce tasks ---
    ("E-com Easy", """Task: Buy a Laptop from the store. Search for it, add to cart, and complete checkout with shipping info: Name=Raj Kumar, Address=12 MG Road, City=Bangalore, PIN=560001, Phone=9876543210

Accessibility Tree:
[ref=mn_1 role=main]
  [ref=frm_1 role=form name="Search Products"]
    [ref=inp_1 role=textbox name="Search" value=""]
    [ref=cmb_1 role=combobox name="Category" value="All"]
    [ref=btn_1 role=button name="Filter"]
  [ref=sec_1 role=region name="Products"]
    [ref=lnk_1 role=link name="Laptop - Premium Build"]
    [ref=lnk_2 role=link name="Headphones - Wireless"]
    [ref=lnk_3 role=link name="Phone Case - Leather"]"""),

    ("E-com Medium", """Task: Buy Headphones from the store. Search, add to cart, checkout. Ship to: Name=Priya Sharma, Address=45 Park Street, City=Kolkata, PIN=700016, Phone=9988776655

Accessibility Tree:
[ref=mn_1 role=main]
  [ref=btn_99 role=button name="No Thanks" description="newsletter popup"]
  [ref=frm_1 role=form name="Product Search"]
    [ref=inp_1 role=textbox name="What are you looking for?" value=""]
    [ref=cmb_1 role=combobox name="Department" value="-- All --"]
    [ref=btn_1 role=button name="Search"]
  [ref=sec_1 role=region name="Results"]
    [ref=lnk_1 role=link name="Headphones - Studio Quality"]
    [ref=btn_98 role=button name="Compare"]
    [ref=lnk_2 role=link name="Speaker - Portable"]"""),

    ("E-com Checkout", """Task: Buy a Laptop from the store. Ship to: Name=Raj Kumar, Address=12 MG Road, City=Bangalore, PIN=560001, Phone=9876543210

Accessibility Tree:
[ref=mn_1 role=main]
  [ref=hdg_1 role=heading name="Checkout"]
  [ref=sec_1 role=region name="Cart Summary"]
    [ref=txt_1 role=text name="Laptop - Premium Build x1"]
  [ref=frm_1 role=form name="Shipping Details"]
    [ref=inp_1 role=textbox name="Full Name" value=""]
    [ref=inp_2 role=textbox name="Address" value=""]
    [ref=inp_3 role=textbox name="City" value=""]
    [ref=inp_4 role=textbox name="PIN Code" value=""]
    [ref=inp_5 role=textbox name="Phone" value=""]
    [ref=btn_1 role=button name="Place Order"]
    [ref=btn_98 role=button name="Save for Later"]"""),
]

print("Post-RL Quality Check (Booking + E-commerce)")
print("=" * 60)

stats = {"json_valid": 0, "action_valid": 0, "has_think": 0, "total": 0,
         "type_actions": 0, "click_actions": 0}
all_outputs = []

for difficulty, test_input in quality_tests:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": test_input},
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    outputs_for_input = []
    for _ in range(3):
        outputs = model.generate(**inputs, max_new_tokens=300, temperature=0.3, do_sample=True)
        response = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()
        outputs_for_input.append(response)
        all_outputs.append(response)
        stats["total"] += 1

        if "<think>" in response:
            stats["has_think"] += 1
        try:
            parsed = parse_model_output(response)
            stats["json_valid"] += 1
            atype = parsed.get("action_type", "")
            if atype in VALID_ACTION_TYPES:
                stats["action_valid"] += 1
            if atype == "type":
                stats["type_actions"] += 1
            elif atype == "click":
                stats["click_actions"] += 1
        except (json.JSONDecodeError, ValueError):
            pass

    unique = len(set(outputs_for_input))
    print(f"\n[{difficulty}]")
    for j, out in enumerate(outputs_for_input):
        think_preview = ""
        if "<think>" in out and "</think>" in out:
            t_start = out.index("<think>") + 7
            t_end = out.index("</think>")
            think_preview = out[t_start:t_end].strip()[:80] + "..."
        action_preview = ""
        if "<answer>" in out:
            a_start = out.index("<answer>") + 8
            action_preview = out[a_start:a_start+80].strip()
        print(f"  {j+1}: think='{think_preview}' action='{action_preview}'")
    print(f"  Diversity: {unique}/3 unique")

t = stats["total"]
unique_total = len(set(all_outputs))
print(f"\n{'=' * 60}")
print(f"Overall: {stats['json_valid']}/{t} valid JSON, {stats['action_valid']}/{t} valid actions")
print(f"Reasoning: {stats['has_think']}/{t} include <think> tags")
print(f"Diversity: {unique_total}/{t} unique outputs")
print(f"Action balance: click={stats['click_actions']}, type={stats['type_actions']}")

if stats["type_actions"] == 0 and stats["click_actions"] > 0:
    print("WARNING: Mode collapse detected — model never outputs 'type'. Check training data balance.")
elif unique_total <= 2:
    print("WARNING: Possible mode collapse — very few unique outputs.")
elif stats["json_valid"] < t * 0.5:
    print("WARNING: JSON validity dropped after RL.")
else:
    print("Model looks healthy after RL training.")

Post-RL Quality Check (Booking + E-commerce)

[Booking Easy]
  1: think='I've filled From (inp_1) and To (inp_2). Selecting "Sleeper" (cmb_1) → Clicking ...' action='{"action_type":"click","element_ref":"btn_1","text_value":"","scroll_delta":0}'
  2: think='I've filled From (inp_1) and To (inp_2). Selecting "Sleeper" (cmb_1) first. Then...' action='{"action_type":"click","element_ref":"btn_1","text_value":"","scroll_delta":0}'
  3: think='I've filled From (inp_1) and To (inp_2). Selecting "Sleeper" (cmb_1) first. Then...' action='{"action_type":"click","element_ref":"btn_1","text_value":"","scroll_delta":0}'
  Diversity: 2/3 unique

[Booking Medium]
  1: think='I've filled Starting Point (inp_1) and Going To (inp_2). cmb_1 is empty (role=co...' action='{"action_type":"type","element_ref":"inp_1","text_value":"Hyderabad","scroll_de'
  2: think='I've filled Starting Point (inp_1) and Going To (inp_2). cmb_1 is empty (role=co...' action='{"action_type":"type","element_ref":"cmb_1","text_v

KeyboardInterrupt: 

In [ ]:
# Cell 10 — Live Environment Evaluation (FIXED)
#
# ROOT CAUSE OF FAILURE:
#   The old version used raw `websockets` library with hand-rolled JSON protocol.
#   But the server uses OpenEnv's `create_app()` which speaks a DIFFERENT protocol.
#   Result: WebSocket handshake fails, connection rejected, "ws closed" errors.
#
# FIX:
#   Use the REST endpoints that create_app() also generates:
#     POST /reset  {"task_id": 1, "seed": 42}  → observation JSON
#     POST /step   {"action_type": "click", "element_ref": "btn_1", ...} → observation JSON
#   These work through HF Spaces proxy with zero protocol issues.
#
# ALSO FIXED:
#   - Added missing `parse_model_output()` function (was called but never defined!)
#   - Proper timeout handling for Playwright cold-start (first reset takes 30-60s)
#   - Handles both wrapped {"observation": {...}} and unwrapped response formats

import json
import os
import time
import httpx

INFINITE_DOM_URL = os.environ.get("INFINITE_DOM_URL", HF_SPACE_URL)

# ── Health check ──
print(f"Environment: {INFINITE_DOM_URL}")
try:
    r = httpx.get(f"{INFINITE_DOM_URL}/health", timeout=30)
    r.raise_for_status()
    print(f"Server status: {r.json()}")
except Exception as e:
    print(f"ERROR: Cannot connect: {e}")
    print("Make sure your HF Space is running (visit the URL in browser to wake it).")
    raise SystemExit("Environment not available")

# ── Fetch task list ──
TASK_LABELS_LIVE = {}
try:
    r = httpx.get(f"{INFINITE_DOM_URL}/tasks", timeout=10)
    if r.status_code == 200:
        for t in r.json().get("tasks", []):
            TASK_LABELS_LIVE[t["task_id"]] = t["description"]
        print(f"Loaded {len(TASK_LABELS_LIVE)} task descriptions")
except Exception:
    pass


# ── Missing function that the old Cell 10 called but never defined ──
def parse_model_output(text):
    """Parse <think>...</think><answer>JSON</answer> into action dict."""
    text = text.strip()
    # Extract from <answer> tags
    if "<answer>" in text:
        start = text.index("<answer>") + len("<answer>")
        end = text.index("</answer>") if "</answer>" in text else len(text)
        text = text[start:end].strip()
    # Handle markdown code blocks
    if text.startswith("```"):
        parts = text.split("```")
        if len(parts) > 1:
            text = parts[1].lstrip("json").strip()
    parsed = json.loads(text)
    return {
        "action_type": parsed.get("action_type", "wait"),
        "element_ref": parsed.get("element_ref", ""),
        "text_value": parsed.get("text_value", ""),
        "scroll_delta": int(parsed.get("scroll_delta", 0)),
    }


# ── REST-based environment client (replaces broken WebSocket code) ──
class RESTEnvClient:
    """Talk to the OpenEnv server over plain HTTP POST.
    
    Why this works when WebSocket didn't:
      - create_app() generates REST endpoints alongside WebSocket
      - HTTP goes through HF Spaces proxy cleanly (no upgrade negotiation)
      - Same payload format the smoke_test.py already validates
    """
    def __init__(self, base_url: str, timeout: float = 120.0):
        self.base_url = base_url.rstrip("/")
        self.timeout = timeout
        self.client = httpx.Client(timeout=httpx.Timeout(timeout, connect=30.0))

    def reset(self, task_id: int = 1, seed: int = 42) -> dict:
        """POST /reset → observation dict."""
        r = self.client.post(
            f"{self.base_url}/reset",
            json={"task_id": task_id, "seed": seed},
        )
        r.raise_for_status()
        data = r.json()
        # Handle both wrapped {"observation": {...}} and unwrapped formats
        if "observation" in data and isinstance(data["observation"], dict):
            obs = data["observation"]
            obs["done"] = data.get("done", False)
            obs["reward"] = data.get("reward", 0)
            return obs
        # Unwrapped: observation fields are at top level
        return data

    def step(self, action: dict) -> dict:
        """POST /step → observation dict."""
        r = self.client.post(
            f"{self.base_url}/step",
            json=action,
        )
        r.raise_for_status()
        data = r.json()
        if "observation" in data and isinstance(data["observation"], dict):
            obs = data["observation"]
            obs["done"] = data.get("done", False)
            obs["reward"] = data.get("reward", 0)
            return obs
        return data

    def close(self):
        self.client.close()


# ── Policies ──
MAX_EVAL_STEPS = 30

def random_policy(obs, step_history=None):
    import random as rng
    return {
        "action_type": rng.choice(["click", "type", "scroll", "wait"]),
        "element_ref": "btn_1",
        "text_value": "test",
        "scroll_delta": 0,
    }


def model_policy(obs, step_history=None):
    """Use trained model with Think-then-Act format + step history."""
    tree = obs.get("a11y_tree", "")[:OBS_MAX_CHARS]
    instruction = obs.get("task_instruction", "")
    progress = obs.get("task_progress", [])

    history_text = ""
    if step_history:
        recent = step_history[-5:]
        history_text = "Previous actions:\n" + "\n".join(f"  {s}" for s in recent) + "\n\n"

    user_msg = (
        f"Task: {instruction}\n\n"
        f"{history_text}"
        f"Accessibility Tree:\n{tree}\n\n"
        f"Completed: {progress}"
    )
    msgs = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_msg},
    ]
    prompt = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    outputs = model.generate(**inputs, max_new_tokens=300, temperature=0.1)
    response = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)

    try:
        return parse_model_output(response)
    except (json.JSONDecodeError, ValueError, KeyError):
        return {"action_type": "wait", "element_ref": "", "text_value": "", "scroll_delta": 0}


# ── Episode runner ──
def run_eval_episode(env_client, policy_fn, task_id=1, seed=42,
                     max_steps=MAX_EVAL_STEPS, debug=False):
    """Run one full episode via REST (reliable, no WebSocket issues)."""
    try:
        obs = env_client.reset(task_id=task_id, seed=seed)

        meta = obs.get("metadata", {})
        total_nodes = meta.get("total_nodes", 5)

        if debug:
            print(f"    [DEBUG] Reset OK | task_instruction: {obs.get('task_instruction', '')[:80]}...")
            print(f"    [DEBUG] total_nodes: {total_nodes}")
            print(f"    [DEBUG] a11y_tree length: {len(obs.get('a11y_tree', ''))}")

        done = obs.get("done", False)
        total_reward = 0.0
        steps = 0
        step_history = []

        while not done and steps < max_steps:
            action = policy_fn(obs, step_history)

            if debug and steps < 3:
                print(f"    [DEBUG] Step {steps}: {action}")

            obs = env_client.step(action)
            done = obs.get("done", False)
            reward = obs.get("reward", 0) or 0
            total_reward += reward
            steps += 1

            # Build step history for context compression
            atype = action.get("action_type", "?")
            ref = action.get("element_ref", "")
            tval = action.get("text_value", "")
            desc = f"Step {steps}: {atype}"
            if ref:
                desc += f" on {ref}"
            if tval:
                desc += f" ('{tval}')"
            step_history.append(desc)

        completed = len(obs.get("task_progress", []))
        return completed, total_nodes, total_reward, steps

    except httpx.TimeoutException:
        print(f"  Timeout (task={task_id}, seed={seed}) — Playwright may be cold-starting")
        return 0, 0, 0.0, 0
    except Exception as e:
        print(f"  Episode failed: {type(e).__name__}: {e}")
        return 0, 0, 0.0, 0


# ══════════════════════════════════════════════════════════════
# RUN EVALUATIONS
# ══════════════════════════════════════════════════════════════
FastLanguageModel.for_inference(model)

env_client = RESTEnvClient(INFINITE_DOM_URL, timeout=120.0)

# ── Quick connectivity test ──
print("\nTesting REST connection with a single reset...")
try:
    test_obs = env_client.reset(task_id=1, seed=999)
    print(f"  REST test: OK — got instruction: {test_obs.get('task_instruction', '')[:60]}...")
except Exception as e:
    print(f"  REST test: FAILED — {e}")
    print("  Check that your HF Space is running and not sleeping.")
    env_client.close()
    raise SystemExit("Cannot connect")

# ── Random baseline ──
print("\n=== Random Baseline (Tasks 1, 5) ===")
random_results = {}
for task_id in [1, 5]:
    nodes_list, totals_list = [], []
    for seed in range(3):
        nodes, total, reward, steps = run_eval_episode(
            env_client, random_policy, task_id=task_id, seed=seed
        )
        nodes_list.append(nodes)
        totals_list.append(total)
    avg = sum(nodes_list) / len(nodes_list)
    avg_total = max(totals_list) if totals_list else 5
    random_results[task_id] = avg
    print(f"  Task {task_id}: avg {avg:.1f}/{avg_total} nodes completed")

# ── Trained model on all 8 tasks ──
print("\n=== Trained Model — All 8 Tasks (Think-then-Act + Step History) ===")
trained_results = {}
trained_totals = {}
DEBUG_MODE = True

for task_id in range(1, 9):
    label = TASK_LABELS_LIVE.get(task_id, TASK_LABELS.get(task_id, f"Task {task_id}"))
    nodes_list, totals_list, steps_list, rewards_list = [], [], [], []

    for seed in range(5):
        debug_this = DEBUG_MODE and seed == 0
        nodes, total, reward, steps = run_eval_episode(
            env_client, model_policy, task_id=task_id, seed=seed, debug=debug_this
        )
        nodes_list.append(nodes)
        totals_list.append(total)
        steps_list.append(steps)
        rewards_list.append(reward)

    avg = sum(nodes_list) / len(nodes_list)
    avg_total = max(totals_list) if totals_list else 5
    avg_steps = sum(steps_list) / len(steps_list)
    avg_reward = sum(rewards_list) / len(rewards_list)
    trained_results[task_id] = avg
    trained_totals[task_id] = avg_total
    print(f"  Task {task_id} ({label}): avg {avg:.1f}/{avg_total} nodes, "
          f"{avg_steps:.0f} steps, reward={avg_reward:.3f}")

# ── Summary ──
print("\n=== Summary ===")
random_avg = sum(random_results.values()) / max(len(random_results), 1)
trained_avg = sum(trained_results.values()) / len(trained_results)
booking_avg = sum(trained_results.get(t, 0) for t in range(1, 5)) / 4
ecom_avg = sum(trained_results.get(t, 0) for t in range(5, 9)) / 4

print(f"Random baseline: ~{random_avg:.1f} nodes avg")
print(f"Trained model:   {trained_avg:.1f} nodes avg (booking: {booking_avg:.1f}, ecommerce: {ecom_avg:.1f})")
print(f"Improvement:     +{trained_avg - random_avg:.1f} nodes")

env_client.close()
print("\nDone. Include these numbers in your README.")

Environment: https://saksham1771-infinite-dom.hf.space
Server status: {'status': 'healthy'}
Loaded 8 task descriptions

Testing REST connection with a single reset...
  REST test: OK — got instruction: Book a AC 1st Class ticket from Mumbai to Lucknow...

=== Random Baseline (Tasks 1, 5) ===
  Episode failed: HTTPStatusError: Client error '422 Unprocessable Entity' for url 'https://saksham1771-infinite-dom.hf.space/step'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/422
  Episode failed: HTTPStatusError: Client error '422 Unprocessable Entity' for url 'https://saksham1771-infinite-dom.hf.space/step'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/422
  Episode failed: HTTPStatusError: Client error '422 Unprocessable Entity' for url 'https://saksham1771-infinite-dom.hf.space/step'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/422
  Task 1: avg 0.0/0 nodes completed
  Episod

In [ ]:
# Cell 11 — Save final model (Guide §16: save LoRA adapters properly)
#
# WARNING: Do NOT upcast 4-bit to 16-bit and merge naively.
# Save LoRA adapters separately — this is the safe, recommended path.

import os

# Save adapters locally
model.save_pretrained("./final_lora_adapters")
tokenizer.save_pretrained("./final_lora_adapters")
print("Saved final LoRA adapters to ./final_lora_adapters")

# Save eval results alongside the model
eval_results = {
    "sft_training_loss": sft_results.training_loss if "sft_results" in dir() else None,
    "reward_history": reward_history if "reward_history" in dir() else [],
    "random_baseline": random_results if "random_results" in dir() else {},
    "trained_results": trained_results if "trained_results" in dir() else {},
    "model_id": MODEL_ID,
    "obs_max_chars": OBS_MAX_CHARS,
}
with open("./final_lora_adapters/eval_results.json", "w") as f:
    json.dump(eval_results, f, indent=2, default=str)
print("Saved eval_results.json alongside adapters")

# Push to HuggingFace Hub
HF_TOKEN = os.environ.get("HF_TOKEN")
HF_REPO = "YOUR_USERNAME/infinite-dom-agent"  # @param {type:"string"}

if HF_TOKEN:
    try:
        model.push_to_hub(HF_REPO, token=HF_TOKEN)
        tokenizer.push_to_hub(HF_REPO, token=HF_TOKEN)
        # Also push eval results
        from huggingface_hub import HfApi
        api = HfApi(token=HF_TOKEN)
        api.upload_file(
            path_or_fileobj="./final_lora_adapters/eval_results.json",
            path_in_repo="eval_results.json",
            repo_id=HF_REPO,
        )
        print(f"Pushed to HuggingFace Hub: {HF_REPO}")
    except Exception as e:
        print(f"Hub push failed: {e}")
        print("Model saved locally — you can push manually later.")
else:
    print("Set HF_TOKEN to push to Hub. Model saved locally.")

In [ ]:
# Cell 12 — Training Plots (save as PNG for README/blog post)
#
# Updated for 8-task curriculum: booking (1-4) + e-commerce (5-8)

import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams["figure.dpi"] = 150

fig, axes = plt.subplots(1, 3, figsize=(20, 5))

# ── Plot 1: SFT Training + Eval Loss ──
if "sft_log_history" in dir() and sft_log_history:
    train_losses = [(h["step"], h["loss"]) for h in sft_log_history if "loss" in h and "eval_loss" not in h]
    eval_losses = [(h["step"], h["eval_loss"]) for h in sft_log_history if "eval_loss" in h]

    if train_losses:
        steps, losses = zip(*train_losses)
        axes[0].plot(steps, losses, "b-", linewidth=1.2, alpha=0.7, label="Train")
    if eval_losses:
        steps, losses = zip(*eval_losses)
        axes[0].plot(steps, losses, "r-", linewidth=1.5, label="Eval")
    axes[0].set_xlabel("Training Step")
    axes[0].set_ylabel("Loss")
    axes[0].set_title("SFT Training & Eval Loss")
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    if eval_losses and len(eval_losses) > 3:
        last_3_eval = [l for _, l in eval_losses[-3:]]
        if last_3_eval[-1] > last_3_eval[0]:
            axes[0].annotate("Potential\noverfitting",
                           xy=(eval_losses[-1][0], eval_losses[-1][1]),
                           fontsize=8, color="red", ha="right")
else:
    axes[0].text(0.5, 0.5, "No SFT data\n(run Cells 6-7 first,\nor resumed from checkpoint)",
                 ha="center", va="center", transform=axes[0].transAxes)

# ── Plot 2: GRPO Reward by Task (8-Task Curriculum) ──
if "reward_history" in dir() and reward_history:
    task_ids = [h["task_id"] for h in reward_history]
    rewards = [h.get("avg_reward") or 0 for h in reward_history]

    # Color gradient: green → yellow → orange → red for each group of 4
    colors_booking = ["#22c55e", "#84cc16", "#eab308", "#ef4444"]
    colors_ecom = ["#06b6d4", "#3b82f6", "#8b5cf6", "#ec4899"]
    colors = []
    for tid in task_ids:
        if tid <= 4:
            colors.append(colors_booking[tid - 1] if tid <= len(colors_booking) else "#888")
        else:
            colors.append(colors_ecom[tid - 5] if tid - 5 < len(colors_ecom) else "#888")

    bars = axes[1].bar(task_ids, rewards, color=colors, edgecolor="black", linewidth=0.5)

    axes[1].set_xlabel("Task ID")
    axes[1].set_ylabel("Avg Reward")
    axes[1].set_title("GRPO Avg Reward by Task (8-Task Curriculum)")
    axes[1].set_xticks(task_ids)

    short_labels = {
        1: "B:Clean", 2: "B:Label", 3: "B:Struct", 4: "B:Chaos",
        5: "E:Clean", 6: "E:Label", 7: "E:Struct", 8: "E:Chaos",
    }
    axes[1].set_xticklabels([short_labels.get(t, str(t)) for t in task_ids],
                             rotation=45, ha="right", fontsize=8)
    axes[1].set_ylim(0, 1.0)
    axes[1].grid(True, alpha=0.3, axis="y")

    for bar, val in zip(bars, rewards):
        axes[1].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.02,
                    f"{val:.2f}", ha="center", va="bottom", fontsize=7)

    # Add group labels
    axes[1].axvline(x=4.5, color="gray", linestyle="--", alpha=0.4)
    axes[1].text(2.5, 0.95, "Booking", ha="center", fontsize=8, color="gray", transform=axes[1].get_xaxis_transform())
    axes[1].text(6.5, 0.95, "E-commerce", ha="center", fontsize=8, color="gray", transform=axes[1].get_xaxis_transform())
else:
    axes[1].text(0.5, 0.5, "No GRPO data\n(run Cell 8 first)",
                 ha="center", va="center", transform=axes[1].transAxes)

# ── Plot 3: Before/After Comparison (All 8 Tasks) ──
if "trained_results" in dir() and trained_results:
    task_ids_eval = sorted(trained_results.keys())
    trained_vals = [trained_results[t] for t in task_ids_eval]
    random_vals = [random_results.get(t, 0) for t in task_ids_eval]

    x = range(len(task_ids_eval))
    width = 0.35
    axes[2].bar([i - width/2 for i in x], random_vals, width,
                label="Random", color="#94a3b8", edgecolor="black", linewidth=0.5)
    axes[2].bar([i + width/2 for i in x], trained_vals, width,
                label="Trained", color="#3b82f6", edgecolor="black", linewidth=0.5)

    axes[2].set_xlabel("Task ID")
    axes[2].set_ylabel("Avg Nodes Completed")
    axes[2].set_title("Random vs Trained Agent (All 8 Tasks)")
    axes[2].set_xticks(list(x))
    axes[2].set_xticklabels([short_labels.get(t, str(t)) for t in task_ids_eval],
                             rotation=45, ha="right", fontsize=8)
    max_nodes = max(trained_totals.values()) if "trained_totals" in dir() and trained_totals else 8
    axes[2].set_ylim(0, max_nodes + 1)
    axes[2].legend()
    axes[2].grid(True, alpha=0.3, axis="y")

    axes[2].axvline(x=3.5, color="gray", linestyle="--", alpha=0.4)
else:
    axes[2].text(0.5, 0.5, "No eval data\n(run Cell 10 first)",
                 ha="center", va="center", transform=axes[2].transAxes)

plt.tight_layout()
plt.savefig("training_curves.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved training_curves.png — include in README and blog post")

In [ ]:
# Cell 13 — Summary Report (8-Task Curriculum)

print("=" * 60)
print("  INFINITE DOM — TRAINING SUMMARY")
print("=" * 60)

print(f"\nBase Model: {MODEL_ID}")
print(f"Quantization: QLoRA 4-bit")
print(f"LoRA Rank: 16")
print(f"Observation Window: {OBS_MAX_CHARS} chars")
print(f"Tasks: 8 (4 booking + 4 e-commerce)")

print(f"\n--- SFT Phase ---")
if "sft_results" in dir():
    print(f"Training loss: {sft_results.training_loss:.4f}")
else:
    print("Training loss: N/A (resumed from checkpoint)")
if "sft_best_metric" in dir() and sft_best_metric is not None:
    print(f"Best eval loss: {sft_best_metric:.4f}")

print(f"\n--- GRPO Phase (8-Task Curriculum) ---")
if "reward_history" in dir() and reward_history:
    print(f"{'Task':>6s}  {'Label':<40s}  {'Loss':>8s}  {'Reward':>8s}")
    print("-" * 70)
    for rh in reward_history:
        r_str = f"{rh['avg_reward']:.4f}" if rh.get("avg_reward") else "N/A"
        print(f"{rh['task_id']:>6d}  {rh['label']:<40s}  {rh['loss']:>8.4f}  {r_str:>8s}")

    booking_rewards = [rh["avg_reward"] for rh in reward_history if rh["task_id"] <= 4 and rh.get("avg_reward")]
    ecom_rewards = [rh["avg_reward"] for rh in reward_history if rh["task_id"] > 4 and rh.get("avg_reward")]
    if booking_rewards:
        print(f"\n  Booking avg reward:    {sum(booking_rewards)/len(booking_rewards):.4f}")
    if ecom_rewards:
        print(f"  E-commerce avg reward: {sum(ecom_rewards)/len(ecom_rewards):.4f}")
else:
    print("No GRPO data (run Cell 8 first)")

if "trained_results" in dir() and trained_results:
    print(f"\n--- Live Evaluation (All 8 Tasks) ---")
    print(f"{'Task':>6s}  {'Nodes':>8s}  {'Random':>8s}  {'Delta':>8s}")
    print("-" * 40)
    for tid in sorted(trained_results.keys()):
        avg = trained_results[tid]
        total = trained_totals.get(tid, 5) if "trained_totals" in dir() else 5
        random_avg = random_results.get(tid, 0) if "random_results" in dir() else 0
        delta = avg - random_avg
        print(f"{tid:>6d}  {avg:>5.1f}/{total:<2d}  {random_avg:>8.1f}  {'+' if delta >= 0 else ''}{delta:>7.1f}")

    trained_avg_all = sum(trained_results.values()) / len(trained_results)
    booking_eval = [trained_results[t] for t in range(1, 5) if t in trained_results]
    ecom_eval = [trained_results[t] for t in range(5, 9) if t in trained_results]
    print(f"\n  Overall avg:    {trained_avg_all:.1f} nodes")
    if booking_eval:
        print(f"  Booking avg:    {sum(booking_eval)/len(booking_eval):.1f} nodes")
    if ecom_eval:
        print(f"  E-commerce avg: {sum(ecom_eval)/len(ecom_eval):.1f} nodes")

print(f"\n--- Submission Checklist ---")
print(f"[ ] HF Space deployed and running")
print(f"[ ] Training notebook runs end-to-end in Colab")
print(f"[ ] LoRA adapters pushed to HF Hub")
print(f"[ ] training_curves.png saved for README/blog")
print(f"[ ] README updated with real results")
print(f"[ ] Blog post written on HuggingFace")
print(f"\n{'=' * 60}")